# Step 2 — Data Cleaning and Harmonization
**Course:** ETL (G01) — Workshop 3  
**Goal:** Standardize all datasets into a single unified schema ready for machine learning.

### Cleaning decisions (justified)
| Decision | Justification |
|----------|---------------|
| Drop `Region` | Only present in 2015 and 2016. Not recoverable for other years without external data. Not a model feature. |
| Drop `Happiness Rank / Overall rank` | Rank is derived from score — using it would cause target leakage. |
| Drop `Standard Error`, `Dystopia Residual`, `Whisker.high/low`, `Confidence Intervals` | Auxiliary columns not shared across years. Not needed for ML. |
| Fill 1 null in `corruption` (2018) | Only 1 missing value out of 156. Filling with median avoids data loss. |
| Add `year` column | Required to identify source year after merging and for the Kafka event schema. |
| Rename all columns to snake_case | Consistent naming across all years. Required for the unified schema. |

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)

## 2. Load Raw Datasets

In [ ]:
# ------------------------------------------------------------------
# Load each year from the raw folder
# ------------------------------------------------------------------
df_2015 = pd.read_csv('../data/raw/2015.csv')
df_2016 = pd.read_csv('../data/raw/2016.csv')
df_2017 = pd.read_csv('../data/raw/2017.csv')
df_2018 = pd.read_csv('../data/raw/2018.csv')
df_2019 = pd.read_csv('../data/raw/2019.csv')

print('Datasets loaded successfully.')

## 3. Define Unified Schema

In [ ]:
# ------------------------------------------------------------------
# Unified schema: these are the final column names after harmonization
# Each year maps its original column names to this unified set
# ------------------------------------------------------------------
UNIFIED_SCHEMA = [
    'country',
    'year',
    'happiness_score',
    'gdp',
    'family',
    'health',
    'freedom',
    'generosity',
    'corruption',
]

# ------------------------------------------------------------------
# Column rename mapping per year
# Original column name -> unified name
# ------------------------------------------------------------------
rename_map = {
    '2015': {
        'Country':                      'country',
        'Happiness Score':              'happiness_score',
        'Economy (GDP per Capita)':     'gdp',
        'Family':                       'family',
        'Health (Life Expectancy)':     'health',
        'Freedom':                      'freedom',
        'Generosity':                   'generosity',
        'Trust (Government Corruption)':'corruption',
    },
    '2016': {
        'Country':                      'country',
        'Happiness Score':              'happiness_score',
        'Economy (GDP per Capita)':     'gdp',
        'Family':                       'family',
        'Health (Life Expectancy)':     'health',
        'Freedom':                      'freedom',
        'Generosity':                   'generosity',
        'Trust (Government Corruption)':'corruption',
    },
    '2017': {
        'Country':                          'country',
        'Happiness.Score':                  'happiness_score',
        'Economy..GDP.per.Capita.':         'gdp',
        'Family':                           'family',
        'Health..Life.Expectancy.':         'health',
        'Freedom':                          'freedom',
        'Generosity':                       'generosity',
        'Trust..Government.Corruption.':    'corruption',
    },
    '2018': {
        'Country or region':            'country',
        'Score':                        'happiness_score',
        'GDP per capita':               'gdp',
        'Social support':               'family',
        'Healthy life expectancy':      'health',
        'Freedom to make life choices': 'freedom',
        'Generosity':                   'generosity',
        'Perceptions of corruption':    'corruption',
    },
    '2019': {
        'Country or region':            'country',
        'Score':                        'happiness_score',
        'GDP per capita':               'gdp',
        'Social support':               'family',
        'Healthy life expectancy':      'health',
        'Freedom to make life choices': 'freedom',
        'Generosity':                   'generosity',
        'Perceptions of corruption':    'corruption',
    },
}

print('Rename maps defined.')

## 4. Cleaning Function

In [ ]:
def clean_dataset(df: pd.DataFrame, year: str, rename: dict) -> pd.DataFrame:
    """
    Clean and harmonize a single year dataset.

    Steps:
    1. Rename columns to unified schema
    2. Keep only unified schema columns
    3. Add year column
    4. Fill missing values with column median
    5. Enforce correct data types
    6. Strip whitespace from country names
    """
    df = df.copy()

    # Step 1 — Rename columns
    df = df.rename(columns=rename)

    # Step 2 — Keep only unified columns (drop rank, region, residual, etc.)
    available = [col for col in UNIFIED_SCHEMA if col != 'year']
    df = df[available]

    # Step 3 — Add year
    df['year'] = int(year)

    # Step 4 — Fill nulls with median (only numeric columns)
    numeric_cols = df.select_dtypes(include='number').columns
    for col in numeric_cols:
        if df[col].isnull().sum() > 0:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            print(f'  [{year}] Filled {col} nulls with median = {median_val:.4f}')

    # Step 5 — Enforce data types
    df['year'] = df['year'].astype(int)
    df['happiness_score'] = df['happiness_score'].astype(float)
    df['gdp']             = df['gdp'].astype(float)
    df['family']          = df['family'].astype(float)
    df['health']          = df['health'].astype(float)
    df['freedom']         = df['freedom'].astype(float)
    df['generosity']      = df['generosity'].astype(float)
    df['corruption']      = df['corruption'].astype(float)

    # Step 6 — Strip whitespace from country names
    df['country'] = df['country'].str.strip()

    # Reorder columns to match unified schema
    df = df[UNIFIED_SCHEMA]

    return df

print('clean_dataset() function defined.')

## 5. Apply Cleaning to Each Year

In [ ]:
# ------------------------------------------------------------------
# Apply the cleaning function to all years
# ------------------------------------------------------------------
raw_datasets = {
    '2015': df_2015,
    '2016': df_2016,
    '2017': df_2017,
    '2018': df_2018,
    '2019': df_2019,
}

cleaned = {}

for year, df in raw_datasets.items():
    print(f'\nCleaning {year}...')
    cleaned[year] = clean_dataset(df, year, rename_map[year])
    print(f'  Shape: {cleaned[year].shape}')
    print(f'  Nulls: {cleaned[year].isnull().sum().sum()}')

print('\nAll datasets cleaned.')

## 6. Validate Unified Schema

In [ ]:
# ------------------------------------------------------------------
# Confirm all years share exactly the same columns and dtypes
# ------------------------------------------------------------------
print('Schema validation:\n')
for year, df in cleaned.items():
    cols_ok   = list(df.columns) == UNIFIED_SCHEMA
    nulls_ok  = df.isnull().sum().sum() == 0
    dupes_ok  = df.duplicated().sum() == 0
    print(f'{year} | columns match: {cols_ok} | no nulls: {nulls_ok} | no dupes: {dupes_ok}')
    if not cols_ok:
        print(f'  Expected: {UNIFIED_SCHEMA}')
        print(f'  Got:      {list(df.columns)}')

In [ ]:
# ------------------------------------------------------------------
# Preview each cleaned dataset
# ------------------------------------------------------------------
for year, df in cleaned.items():
    print(f'\n========== {year} ==========')
    display(df.head(3))
    print(df.dtypes)

## 7. Merge into Unified Dataset

In [ ]:
# ------------------------------------------------------------------
# Concatenate all cleaned years into a single dataframe
# ------------------------------------------------------------------
df_unified = pd.concat(cleaned.values(), ignore_index=True)

print(f'Unified dataset shape: {df_unified.shape}')
print(f'Years present: {sorted(df_unified["year"].unique())}')
print(f'Total nulls: {df_unified.isnull().sum().sum()}')
print(f'Total duplicates: {df_unified.duplicated().sum()}')
display(df_unified.head())

## 8. Post-Merge Quality Check

In [ ]:
# ------------------------------------------------------------------
# Records per year in the unified dataset
# ------------------------------------------------------------------
print('Records per year:')
display(df_unified['year'].value_counts().sort_index())

In [ ]:
# ------------------------------------------------------------------
# Descriptive statistics of the unified dataset
# ------------------------------------------------------------------
display(df_unified.describe().round(3))

In [ ]:
# ------------------------------------------------------------------
# Distribution of happiness_score in unified dataset
# ------------------------------------------------------------------
plt.figure(figsize=(10, 4))
sns.histplot(df_unified['happiness_score'], bins=30, kde=True, color='steelblue')
plt.title('Happiness Score Distribution — Unified Dataset (2015–2019)')
plt.xlabel('Happiness Score')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Correlation heatmap on the unified dataset
# ------------------------------------------------------------------
numeric_cols = ['happiness_score', 'gdp', 'family', 'health', 'freedom', 'generosity', 'corruption']
corr = df_unified[numeric_cols].corr().round(2)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Correlation Heatmap — Unified Dataset')
plt.tight_layout()
plt.show()

## 9. Save Unified Dataset

In [ ]:
# ------------------------------------------------------------------
# Save the unified cleaned dataset to the processed folder
# This file will be used in model_training.ipynb
# ------------------------------------------------------------------
output_path = '../data/processed/happiness_unified.csv'
df_unified.to_csv(output_path, index=False)

print(f'Unified dataset saved to: {output_path}')
print(f'Shape: {df_unified.shape}')